<a href="https://colab.research.google.com/github/Sai-charan822/CSA0732-COMPUTER-NETWORKS/blob/main/LAB/EXP12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Earley Parser Implementation

An Earley parser parses context-free grammars. For every input token, it generates a state set $S_i$. Each state in a state set is represented as an **Earley State (or Item)**:
`[A -> ̑ . ̒, i]` where:
- `A -> ̑ ̒` is a production rule.
- The dot `.` represents the current position in the parsing process.
- `i` is the starting index in the input where this production rule began matching.

In [1]:
class State:
    def __init__(self, rule_name, rule_body, dot_index, start_index):
        self.rule_name = rule_name
        self.rule_body = rule_body
        self.dot_index = dot_index
        self.start_index = start_index

    @property
    def is_completed(self):
        return self.dot_index >= len(self.rule_body)

    @property
    def next_symbol(self):
        if self.is_completed:
            return None
        return self.rule_body[self.dot_index]

    def __eq__(self, other):
        return (self.rule_name == other.rule_name and
                self.rule_body == other.rule_body and
                self.dot_index == other.dot_index and
                self.start_index == other.start_index)

    def __hash__(self):
        return hash((self.rule_name, tuple(self.rule_body), self.dot_index, self.start_index))

    def __repr__(self):
        body_str = list(self.rule_body)
        body_str.insert(self.dot_index, '.')
        return f"[{self.rule_name} -> {' '.join(body_str)}, {self.start_index}]"

In [2]:
class EarleyParser:
    def __init__(self, grammar, start_symbol):
        self.grammar = grammar  # Dict of { NonTerminal: [[symbols], [symbols], ...] }
        self.start_symbol = start_symbol

    def parse(self, tokens):
        # Initialize State Sets
        S = [[] for _ in range(len(tokens) + 1)]

        # Add dummy start rule
        gamma_rule = (f"{self.start_symbol}'", [self.start_symbol])
        S[0].append(State(gamma_rule[0], gamma_rule[1], 0, 0))

        for i in range(len(tokens) + 1):
            state_set = S[i]
            visited_states = set()

            index = 0
            while index < len(state_set):
                state = state_set[index]
                if state in visited_states:
                    index += 1
                    continue
                visited_states.add(state)

                if not state.is_completed:
                    next_sym = state.next_symbol
                    if next_sym in self.grammar: # Non-terminal
                        self.predictor(state, i, S)
                    else: # Terminal
                        if i < len(tokens):
                            self.scanner(state, i, S, tokens[i])
                else:
                    self.completer(state, i, S)

                index += 1

        # Check if the start state was completed successfully spanning the whole input
        target_state = State(f"{self.start_symbol}'", [self.start_symbol], 1, 0)
        return target_state in S[len(tokens)]

    def predictor(self, state, i, S):
        non_terminal = state.next_symbol
        for alternative in self.grammar[non_terminal]:
            new_state = State(non_terminal, alternative, 0, i)
            if new_state not in S[i]:
                S[i].append(new_state)

    def scanner(self, state, i, S, current_token):
        terminal = state.next_symbol
        if terminal == current_token:
            new_state = State(state.rule_name, state.rule_body, state.dot_index + 1, state.start_index)
            if new_state not in S[i + 1]:
                S[i + 1].append(new_state)

    def completer(self, state, i, S):
        for old_state in S[state.start_index]:
            if not old_state.is_completed and old_state.next_symbol == state.rule_name:
                new_state = State(old_state.rule_name, old_state.rule_body, old_state.dot_index + 1, old_state.start_index)
                if new_state not in S[i]:
                    S[i].append(new_state)

### Demo of Earley Parser
We will define a simple arithmetic grammar:
* `E -> E + T | T`
* `T -> T * F | F`
* `F -> ( E ) | number`

In [3]:
grammar = {
    'E': [['E', '+', 'T'], ['T']],
    'T': [['T', '*', 'F'], ['F']],
    'F': [['(', 'E', ')'], ['number']]
}

parser = EarleyParser(grammar, 'E')

# Test input representing "( number + number ) * number"
input_tokens = ['(', 'number', '+', 'number', ')', '*', 'number']
result = parser.parse(input_tokens)
print(f"Parsed tokens {input_tokens}: {result}")

# Invalid input demo
invalid_tokens = ['(', 'number', '+']
result_invalid = parser.parse(invalid_tokens)
print(f"Parsed tokens {invalid_tokens}: {result_invalid}")

Parsed tokens ['(', 'number', '+', 'number', ')', '*', 'number']: True
Parsed tokens ['(', 'number', '+']: False
